In [2]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from implicit.cpu.als import AlternatingLeastSquares

c:\Users\JuanOrtizAlonso\business-aware-recommender-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATA_DIR = Path("../data")

PROCESSED_DIR = DATA_DIR / "processed"

ARTICLES_PATH = PROCESSED_DIR / "articles.parquet"
CUSTOMERS_PATH = PROCESSED_DIR / "customers.parquet"
INTERACTIONS_PATH = PROCESSED_DIR / "interactions.parquet"

In [4]:
articles = pd.read_parquet(ARTICLES_PATH)

customers = pd.read_parquet(CUSTOMERS_PATH)

interactions = pd.read_parquet(INTERACTIONS_PATH)

print(f"Articles: {articles.shape}")
print(f"Customers: {customers.shape}")
print(f"Interactions: {interactions.shape}")

Articles: (99604, 25)
Customers: (100000, 7)
Interactions: (10725535, 3)


In [5]:
user_ids = interactions["customer_id"].unique()
item_ids = interactions["article_id"].unique()

user_to_idx = {
    user: idx
    for idx, user in enumerate(user_ids)
}

item_to_idx = {
    item: idx
    for idx, item in enumerate(item_ids)
}

In [6]:
idx_to_user = {
    idx: user
    for user, idx in user_to_idx.items()
}

idx_to_item = {
    idx: item
    for item, idx in item_to_idx.items()
}

In [7]:
interactions["user_idx"] = (
    interactions["customer_id"]
    .map(user_to_idx)
)

interactions["item_idx"] = (
    interactions["article_id"]
    .map(item_to_idx)
)

In [8]:
user_item_matrix = csr_matrix(
    (
        interactions["interaction"].astype(np.float32),
        (
            interactions["user_idx"],
            interactions["item_idx"]
        )
    )
)

print(user_item_matrix.shape)

(100000, 99604)


In [9]:
model = AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=20,
    random_state=42
)

c:\Users\JuanOrtizAlonso\business-aware-recommender-system\.venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


In [10]:
model.fit(user_item_matrix)

100%|██████████| 20/20 [00:39<00:00,  1.96s/it]


In [11]:
test_user = 0

ids, scores = model.recommend(
    userid=test_user,
    user_items=user_item_matrix[test_user],
    N=10
)

print(ids)
print(scores)

[ 2585  3572  1043  5579   304  1861   690  1375 36545   134]
[0.28750154 0.26821592 0.19955081 0.17026792 0.13152015 0.08299681
 0.07833356 0.07475255 0.070072   0.07001273]


In [12]:
TOP_K = 100
DEMO_USERS = 5000

In [13]:
np.random.seed(42)

selected_users = np.random.choice(
    user_item_matrix.shape[0],
    size=DEMO_USERS,
    replace=False
)

In [14]:
recommendations = []

for i, user_idx in enumerate(selected_users):

    item_ids_rec, scores = model.recommend(
        userid=user_idx,
        user_items=user_item_matrix[user_idx],
        N=TOP_K
    )

    recommendations.append(
        pd.DataFrame({
            "user_idx": user_idx,
            "item_idx": item_ids_rec,
            "score": scores
        })
    )

    if (i + 1) % 500 == 0:
        print(
            f"Processed {i + 1:,}/{DEMO_USERS:,} users"
        )

Processed 500/5,000 users
Processed 1,000/5,000 users
Processed 1,500/5,000 users
Processed 2,000/5,000 users
Processed 2,500/5,000 users
Processed 3,000/5,000 users
Processed 3,500/5,000 users
Processed 4,000/5,000 users
Processed 4,500/5,000 users
Processed 5,000/5,000 users


In [15]:
candidate_recommendations = pd.concat(
    recommendations,
    ignore_index=True
)

In [16]:
candidate_recommendations.shape

(500000, 3)

In [17]:
candidate_recommendations["customer_id"] = (
    candidate_recommendations["user_idx"]
    .map(idx_to_user)
)

candidate_recommendations["article_id"] = (
    candidate_recommendations["item_idx"]
    .map(idx_to_item)
)

In [18]:
product_columns = [
    "article_id",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "colour_group_name"
]

In [19]:
candidate_recommendations = (
    candidate_recommendations
    .merge(
        articles[product_columns],
        on="article_id",
        how="left"
    )
)

In [20]:
candidate_recommendations = (
    candidate_recommendations
    .sort_values(
        ["customer_id", "score"],
        ascending=[True, False]
    )
)

In [21]:
with open(
    PROCESSED_DIR / "user_to_idx.pkl",
    "wb"
) as f:
    pickle.dump(user_to_idx, f)

with open(
    PROCESSED_DIR / "item_to_idx.pkl",
    "wb"
) as f:
    pickle.dump(item_to_idx, f)

with open(
    PROCESSED_DIR / "als_model.pkl",
    "wb"
) as f:
    pickle.dump(model, f)

In [22]:
demo_users = pd.DataFrame({
    "user_idx": selected_users
})

demo_users["customer_id"] = (
    demo_users["user_idx"]
    .map(idx_to_user)
)

demo_users.to_parquet(
    PROCESSED_DIR / "demo_users.parquet",
    index=False
)

In [23]:
candidate_recommendations.to_parquet(
    PROCESSED_DIR / "candidate_recommendations.parquet",
    index=False
)